# 얼굴 생성 — StyleGAN2-ADA 추론 (FFHQ-256 → CelebV-HQ)

NVIDIA의 FFHQ-256 체크포인트를 CelebV-HQ 프레임으로 전이학습(transfer-learning)한
StyleGAN2-ADA 생성기로부터 얼굴 이미지 1,000장을 생성하는 **재현 가능한 추론 코드**.

**진행 순서:** 환경 점검 → 저장소 클론 + 호환성 패치 → 체크포인트 로드 →
1,000장 생성(시드 고정) → `submission.zip` 패키징

**최종 설정:** `truncation_psi = 1.0`

**환경:** RTX 5090 (Blackwell, sm_120) 단일 GPU, PyTorch 2.x, CUDA 12.8+.


## 0. 환경 점검 (PyTorch + CUDA)

In [ ]:
import subprocess

def check_torch():
    """CUDA 사용 가능한 PyTorch인지 확인하고 GPU 연산 능력(compute capability)을 보고."""
    try:
        import torch
        if not torch.cuda.is_available():
            return False, "CUDA 사용 불가"
        cap = torch.cuda.get_device_capability(0)
        # GPU 커널이 실제로 동작하는지 간단한 행렬곱으로 확인
        _ = (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
        return True, f"torch {torch.__version__}, sm_{cap[0]}{cap[1]}, {torch.cuda.get_device_name(0)}"
    except Exception as e:
        return False, str(e)

ok, msg = check_torch()
print("PyTorch OK:", ok, "|", msg)
if not ok:
    raise SystemExit("추론에는 CUDA 지원 PyTorch가 필요합니다.")


## 1. StyleGAN2-ADA 클론 및 호환성 패치

공식 NVlabs 저장소를 클론한 뒤, 2020년 코드베이스가 최신
PyTorch 2.x / NumPy 2 / Blackwell(sm_120) 환경에서 동작하도록 세 가지 패치를 적용.

1. **깨지는 `gradfix` op 비활성화** → native `torch.nn.functional` 연산으로 대체.
2. **NumPy 2에서 제거된 별칭 패치** (`np.float`, `np.int` 등) — 단어 경계 정규식을 사용해
   `np.float32` 같은 유효한 이름은 보존.
3. **`upfirdn2d` / `bias_act`의 reference 구현 강제** (`impl='ref'`) — 불안정한
   JIT NVCC 컴파일을 피함. 다소 느리지만 재현성이 보장됨.


In [ ]:
import os

WORK = "/content/sg"
REPO_DIR = f"{WORK}/stylegan2-ada-pytorch"
os.makedirs(WORK, exist_ok=True)

# --- 저장소 클론 ---
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git {REPO_DIR}

# --- 의존성 설치 (torch / numpy 는 건드리지 않음) ---
!pip -q install ninja click requests tqdm scipy psutil imageio imageio-ffmpeg
!pip -q install huggingface_hub torch-fidelity
!pip -q install pyspng 2>/dev/null || echo "(pyspng optional)"

# --- (a) gradfix 커스텀 op 비활성화 -> native fallback ---
!sed -i 's/^enabled = True/enabled = False/' {REPO_DIR}/torch_utils/ops/conv2d_gradfix.py
!sed -i 's/^enabled = True/enabled = False/' {REPO_DIR}/torch_utils/ops/grid_sample_gradfix.py

# --- (b) NumPy 2 별칭 패치 (단어 경계로 np.float32 등은 보호) ---
!cd {REPO_DIR} && find . -name '*.py' -print0 | xargs -0 sed -i -E \
  's/\bnp\.float\b/np.float64/g; s/\bnp\.int\b/np.int64/g; s/\bnp\.bool\b/np.bool_/g; s/\bnp\.object\b/np.object_/g; s/\bnp\.complex\b/np.complex128/g'

# --- (c) upfirdn2d / bias_act 를 reference 구현으로 강제 (NVCC 불필요) ---
!sed -i "s/impl='cuda'/impl='ref'/g" {REPO_DIR}/torch_utils/ops/upfirdn2d.py {REPO_DIR}/torch_utils/ops/bias_act.py

print("패치 적용 완료.")


## 2. 설정

모든 설정값. `NETWORK_PKL`에 학습된 생성기 스냅샷
(`G_ema`를 포함한 `.pkl`) 경로를 지정. `None`으로 두면 `OUTDIR` 아래에 snapshot 파일 넣으면 자동으로 모델 선택 완료. 재현성을 위해 시드는 고정.


In [ ]:
import torch

# --- 경로 ---
PROJECT_DIR = "/content/face_contest"
OUTDIR      = f"{PROJECT_DIR}/training-runs"   # 학습 스냅샷 위치
SAMPLES_DIR = f"{PROJECT_DIR}/samples_raw"     # 생성된 PNG
SUB_DIR     = f"{PROJECT_DIR}/submission_imgs" # 최종 JPG
SUB_ZIP     = f"{PROJECT_DIR}/submission.zip"  # 제출 아카이브

# --- 체크포인트 (None = OUTDIR 아래 최신 스냅샷 자동 선택) ---
NETWORK_PKL = None

# --- 샘플링 하이퍼파라미터 ---
TRUNCATION_PSI = 1.0     # 최종 선택값. {0.7, 0.8, 1.0} 스윕에서 FID/IS/KID 최상
SEED           = 12345   # 재현성을 위한 고정 시드
NUM_IMAGES     = 1000
BATCH_SIZE     = 25

device = torch.device("cuda")
for d in (SAMPLES_DIR, SUB_DIR, OUTDIR):
    os.makedirs(d, exist_ok=True)
print("device =", torch.cuda.get_device_name(0))


## 3. 학습된 생성기 로드

In [ ]:
import sys, glob
sys.path.insert(0, REPO_DIR)
import dnnlib, legacy

# 체크포인트 경로 결정
if NETWORK_PKL is None:
    snaps = sorted(glob.glob(f"{OUTDIR}/network-snapshot-*.pkl"))
    assert snaps, f"{OUTDIR} 아래에 스냅샷이 없습니다. NETWORK_PKL을 직접 지정하세요."
    NETWORK_PKL = snaps[-1]
print("로드:", NETWORK_PKL)

with dnnlib.util.open_url(NETWORK_PKL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device).eval()
print(f"생성기 로드 완료 | z_dim={G.z_dim}, c_dim={G.c_dim}, img_resolution={G.img_resolution}")


## 4. 이미지 1,000장 생성


In [ ]:
import numpy as np
import PIL.Image
from tqdm.auto import tqdm

# 이전 샘플 제거
for p in glob.glob(f"{SAMPLES_DIR}/*.png"):
    os.remove(p)

idx = 0
with torch.no_grad():
    for start in tqdm(range(0, NUM_IMAGES, BATCH_SIZE), desc="생성 중"):
        bs = min(BATCH_SIZE, NUM_IMAGES - start)
        z = torch.from_numpy(
            np.stack([np.random.RandomState(SEED + start + j).randn(G.z_dim) for j in range(bs)])
        ).to(device).float()
        c = torch.zeros([bs, G.c_dim], device=device)

        img = G(z, c, truncation_psi=TRUNCATION_PSI, noise_mode="const")
        img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()

        for j in range(bs):
            PIL.Image.fromarray(img[j], "RGB").save(f"{SAMPLES_DIR}/img_{idx:04d}.png")
            idx += 1

print("생성 완료:", idx)

# 미리보기
from IPython.display import display
for p in sorted(glob.glob(f"{SAMPLES_DIR}/*.png"))[:4]:
    display(PIL.Image.open(p).resize((160, 160)))


## 5. `submission.zip` 패키징

PNG를 고품질 JPG로 변환하고, 정확히 1,000장으로 구성된 평면(flat) 아카이브를 만든 뒤,
재현성을 위한 샘플링 메타데이터를 기록.


In [ ]:
import zipfile, json
from PIL import Image

# PNG -> JPG 변환
for p in glob.glob(f"{SUB_DIR}/*.jpg"):
    os.remove(p)
for i in tqdm(range(NUM_IMAGES), desc="JPG 변환"):
    Image.open(f"{SAMPLES_DIR}/img_{i:04d}.png").convert("RGB").save(
        f"{SUB_DIR}/img_{i:04d}.jpg", "JPEG", quality=95, subsampling=0)

# zip 생성
if os.path.exists(SUB_ZIP):
    os.remove(SUB_ZIP)
with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for i in range(NUM_IMAGES):
        zf.write(f"{SUB_DIR}/img_{i:04d}.jpg", arcname=f"img_{i:04d}.jpg")

# 검증
size_mb = os.path.getsize(SUB_ZIP) / 1e6
with zipfile.ZipFile(SUB_ZIP) as zf:
    names = zf.namelist()
assert len(names) == NUM_IMAGES, f"{NUM_IMAGES}장이어야 하는데 {len(names)}장입니다"
assert all("/" not in n for n in names), "아카이브는 평면 구조여야 합니다(하위 폴더 불가)"
assert size_mb < 200, f"아카이브가 너무 큽니다: {size_mb:.1f} MB"

# 메타데이터
meta = dict(
    method="StyleGAN2-ADA transfer ffhq256->CelebV-HQ",
    cfg="paper256",
    network=NETWORK_PKL,
    truncation_psi=TRUNCATION_PSI,
    sample_seed=SEED,
    num_images=NUM_IMAGES,
)
with open(f"{PROJECT_DIR}/submission_meta.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"{SUB_ZIP}: {size_mb:.1f} MB | {len(names)} 파일")
print("메타데이터:", json.dumps(meta, indent=2, ensure_ascii=False))
